### 가상환경 check

In [ ]:
import sys
import os

print(f"Python version: {sys.version}")
print(f"Executable:     {sys.executable}")
print(f"Prefix:         {sys.prefix}")
# python 환경의 root directory
print(f"In virtualenv:  {sys.prefix != sys.base_prefix}")
# 가상환경 여부 판단: 현재 python 환경 & 원래 python(system python)
# 가상 환경과 시스템 환경 다른지 여부 검사 통해 가상환경 접속 여부 test

Python version: 3.13.12 (main, Mar 20 2026, 00:20:47) [Clang 22.1.1 ]
Executable:     /Users/carolyn/Desktop/study/데이터 자동화/practice/BDAI_DataAutomationHandling/.venv/bin/python
Prefix:         /Users/carolyn/Desktop/study/데이터 자동화/practice/BDAI_DataAutomationHandling/.venv
In virtualenv:  True


### package version check

In [3]:
import importlib.metadata
# 설치된 python package의 metadata를 조회하는 모듈

PACKAGES = [
    "pandas",
    "pyarrow",
    "google-cloud-bigquery",
    "google-cloud-bigquery-storage",
    "db-dtypes",
    "matplotlib",
    "requests",
    "tqdm",
    "ipykernel",
    "nbconvert",
]

print(f"{'Package':<35s} {'Version':<15s}")
print("-"*50)
for pkg in PACKAGES:
    try:
        ver = importlib.metadata.version(pkg)
        print(f"{pkg:<35s} {ver:<15s}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:<35s} {'NOT INSTALLED':<15s}")

Package                             Version        
--------------------------------------------------
pandas                              3.0.1          
pyarrow                             23.0.1         
google-cloud-bigquery               3.40.1         
google-cloud-bigquery-storage       2.36.2         
db-dtypes                           1.4.4          
matplotlib                          3.10.8         
requests                            2.32.5         
tqdm                                4.67.3         
ipykernel                           7.2.0          
nbconvert                           7.17.0         


pyproject.toml=project의 의도/설정서\
uv.lock=실제 설치에 사용할 확정 결과물/잠금 파일

### uv check

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path("../")
pyproject = PROJECT_ROOT / "pyproject.toml"
assert pyproject.exists(), "project.toml not found"
print("pyproject.toml:")
print(pyproject.read_text())

python_version_file = PROJECT_ROOT / ".python-version"
if python_version_file.exists():
    print(f"\n.python-version: {python_version_file.read_ext().strip()}")

uv_lock = PROJECT_ROOT / "uv.lock"
print(f"\nuv.lock exists: {uv_lock.exists()}")
if uv_lock.exists():
    print(f"uv.lock size: {uv_lock.stat().st_size / 1024:.1f} KB")
    # byte -> kb
    # stat(): file의 metadata를 가져오는 함수

pyproject.toml:
[project]
name = "bda-2"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
requires-python = ">=3.13"
dependencies = [
    "db-dtypes>=1.4.4",
    "google-cloud-bigquery>=3.40.1",
    "google-cloud-bigquery-storage>=2.36.2",
    "implicit>=0.7.2",
    "ipykernel>=7.2.0",
    "matplotlib>=3.10.8",
    "nbconvert>=7.17.0",
    "pandas>=3.0.1",
    "pyarrow>=23.0.1",
    "requests>=2.32.5",
    "scipy>=1.17.1",
    "tqdm>=4.67.3",
]

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[tool.hatch.build.targets.wheel]
packages = ["src/gharchive", "src/ghrec"]


uv.lock exists: True
uv.lock size: 270.5 KB


### GCP service account & BigQuery connection check

In [8]:
pwd

'/Users/carolyn/Desktop/study/데이터 자동화/practice/BDAI_DataAutomationHandling/gharchive'

In [9]:
os.chdir("..")

In [10]:
import json
KEY_PATH = os.environ.get("GCP_KEY_PATH", "gcp-key.json")
key_file = Path(KEY_PATH)
print(f"Key path: {KEY_PATH}")
print(f"Key exists: {key_file.exists()}")

Key path: gcp-key.json
Key exists: True


In [ ]:
if key_file.exists():
    with open(key_file) as f:
        key_data = json.load(f)
    print(f"Project ID: {key_data.get('project_id', 'N/A')}")
    print(f"Client email: {key_data.get('client_email', 'N/A')}")
    print(f"Key type: {key_data.get('type', 'N/A')}")
else:
    print("\n⚠ gcp-key.json이 없습니다. BigQuery 노트북(01, ghrec/01, ghrec/03)을 실행할 수 없습니다.")
    print("GCP Console에서 서비스 계정 키를 다운로드하세요.")

Project ID: gen-lang-client-0791667039
Client email: carolyn@gen-lang-client-0791667039.iam.gserviceaccount.com
Key type: service_account


In [48]:
# Big Query Connection test
from client import create_client

if key_file.exists():
    client = create_client(KEY_PATH)
    print(f"BigQuery client ready")
    print(f"Project: {client.project}")

    test_query = "Select 1 as test"
    result = client.query(test_query).to_dataframe()
    # query로 바다온 데이터는 queryjob 객체일 뿐
    # result는 result.result _ 반복 가능한 row data (row iterator)
    # 분석 불편, 시각화 어려움, ML 어려움
    # so to dataframe
    print(f"Test query OK: {result['test'].iloc[0]}")
else:
    print("no key file")

BigQuery client ready
Project: gen-lang-client-0791667039
Test query OK: 1


### Github Token

In [49]:
import subprocess
#  python code 안에서 OS 명령어를 실행하는 도구
result = subprocess.run(["gh", "auth", "token"], capture_output=True, text=True)
gh_token = result.stdout.strip()
if gh_token:
    print(f"gh CLI token: set ({gh_token[:8]}...)")
else:
    print("gh CLI token: not set")
    print("  → gh auth login 으로 설정하거나")
    print("  → GITHUB_TOKEN 환경변수를 설정하세요")

gh CLI token: set (gho_0xjH...)
